Description: Get peak information, perform coincidence, and plot summed spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor
from application.peak_processor import Peak_Processor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


### Import and Process Data

In [ ]:
df_SPE_position = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spe_position_LXe_2.csv",
                 delimiter=",")

df_SPE_position

In [ ]:
peak_processor = Peak_Processor()


df = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


# convert string to array
if isinstance(df['board_0_channels'][0], str):
    if "," in df['board_0_channels'][0]:
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)
    else:
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)

        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(json.loads).apply(np.array)

print(len(df))
idx_nan = np.where((df['spe_position'].isna()) & (df['voltage_preamp1_V']<=-46))[0]
# replace NaN values with SPE position
voltage_array = df.loc[idx_nan, 'voltage_preamp1_V']
channel_array = df.loc[idx_nan, 'channel']
print(len(idx_nan))
for idx, voltage, channel in zip(idx_nan, voltage_array, channel_array):
    # find the SPE position for the given voltage and channel
    df.loc[idx, 'spe_position'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position'].values
    df.loc[idx, 'spe_position_err'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position_err'].values


In [ ]:
#### Basic Data Selection

all_runs_d2d = d2d.data(df)
mask_data_taking_mode = (all_runs_d2d.data_taking_mode == "all_channels")
mask_na = ~np.isnan(all_runs_d2d.spe_position)
# mask = ~np.isnan(all_runs_d2d.gain)
# mask_voltage = (all_runs_d2d.voltage_preamp1_V == -47)
# mask_run_tag = (all_runs_d2d.run_tag == "LXe/Cs137")
mask_run_tag = (all_runs_d2d.run_tag == "LXe/gain_calibration")
mask = mask_data_taking_mode & mask_na & mask_run_tag

all_runs_d2d.apply_mask(mask, inplace=True, dry = False)
all_run_list = np.unique(all_runs_d2d.md_full_path)

#### Run Selection

check how many runs have all 24 channel data, print a list

In [ ]:
count_successful = 0
count_failed = 0

count_n_events = 0

for i, md_full_path in enumerate(all_run_list):

    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = True)

    if len(single_run.channel) < 24:
        # print(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels. Run tag: {single_run.run_tag[0]}. Comment: {single_run.comment[0]}. Voltage: {single_run.voltage_preamp1_V[0]}")
        count_failed += 1

        continue    
    elif len(single_run.channel) == 24:
        print(f"Run {i}: {md_full_path} has 24 channels. Run tag: {single_run.run_tag[0]}. Voltage: {single_run.voltage_preamp1_V[0]}. Comment: {single_run.comment[0]}. Event number: {single_run.n_processed_events[0]}")
        count_successful += 1
        count_n_events += single_run.n_processed_events[0]
        
    else: 
        raise ValueError(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")

print(f"Total runs: {len(all_run_list)}"
      f"Successful runs: {count_successful} "
      f"Failed runs: {count_failed} "
      f"Success rate: {count_successful/len(all_run_list)*100:.2f}%")

print(f"Total number of events: {count_n_events}")

#### Process Run

In [ ]:
### Choose a run from the list above
run_id = 0

# initialize the result storage
df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
single_info_list = []

for md_full_path in all_run_list[run_id:run_id+1]:
    result, waveform, baseline, baseline_std = peak_processor.get_peak_level_data(
        all_runs_d2d=all_runs_d2d,
        md_full_path=md_full_path,
        peak_merge_window_sample=250
    )
    single_info_list += result

df_result = pd.DataFrame.from_dict(single_info_list)
d2d_data = d2d.data(df_result)



### Check saturation

In [ ]:
for channel in range(24):
    mask = d2d_data.channel == channel
    singl_channel_data = d2d_data.apply_mask(mask, inplace=False, dry=True)

    height_V = singl_channel_data.peak_height_V

    plt.hist(singl_channel_data.peak_height_V, alpha = 0.5, bins=100, range=[1,1.5], label=f"Channel {channel}")

plt.xlabel("Peak Height [V]")
plt.ylabel("Counts")
plt.legend(ncols=2, loc='upper right', fontsize='small')

### Check Waveforms

In [ ]:
mask = (d2d_data.peak_height_V > 1.1) & (d2d_data.channel == 14)
# mask = (d2d_data.peak_area_PE > 1e100) 
# mask = (d2d_data.peak_width_ns > 300) & (d2d_data.peak_height_V > 0.4) & (d2d_data.channel == 14)
# mask = (d2d_data.peak_width_ns > 400) & (d2d_data.peak_width_ns < 750) & (d2d_data.peak_width_ns > 500) & (d2d_data.channel == 11)
# mask = (d2d_data.peak_height_V > 1.2) & (d2d_data.channel == 11)
selected_data = d2d_data.apply_mask(mask, inplace=False, dry=True)
len(selected_data)

single_data = selected_data.get_row_info_to_dict(0)
single_info = WaveformInfo()
single_info.set_info_from_dict(single_data)
waveform, baseline, baseline_std = peak_processor.get_waveform_from_single_info(single_info)

event_id_array = selected_data.event_id


In [ ]:
# can specify the event_id here
# event_id = 6564
event_id = None

if event_id is None:
    event_id_id = np.random.randint(0, len(event_id_array), size=1)[0]
    event_id = event_id_array[event_id_id]

print(f"Selected event_id: {event_id}")

single_waveform = waveform[event_id,:]
single_baseline = baseline[event_id]
single_baseline_std = baseline_std[event_id]
threshold_sig=3
extend_sum_window=50
peak_merge_window_sample = 250 # samples
        
window_size = 6
min_peak_width_sample = window_size*3

single_info.set_peaks_for_single_processed_waveform(
                    single_waveform, 
                    single_baseline, 
                    single_baseline_std,
                    threshold_sig=5, 
                    peak_merge_window_sample=peak_merge_window_sample,
                    show_plot = True,
                    event_id=event_id
                )


# fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/example_waveform-{event_id}.pdf"
# plt.savefig(fname, dpi=300, bbox_inches='tight')
